# 05 — LLM Cluster Labels

**input:** `data/movies.pkl`, `data/embeddings.npy`

**Does one thing:**
Sends a few representative movies per cluster to GPT
and retrieves a short, natural-language label.

Example:
```
Toy Story, A Bug's Life, Shrek  →  'Animated family adventures'
The Godfather, Goodfellas       →  'Gritty crime dramas'
```

**setup:** create a `.env` file next to the notebooks with the following content:
```
OPENAI_API_KEY=sk-...
```

In [ ]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from sklearn.cluster import KMeans
from openai import OpenAI
from dotenv import load_dotenv
import os

# load API key from .env file
load_dotenv(Path('../.env'))
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

DATA_DIR = Path('../data')
movies     = pd.read_pickle(DATA_DIR / 'movies.pkl')
embeddings = np.load(DATA_DIR / 'embeddings.npy')

print('imports OK')
print(f'API key loaded: {bool(os.getenv("OPENAI_API_KEY"))}')

imports OK
API key loaded: True


### Part 1 — label generation function

In [ ]:
def generate_cluster_label(cluster_indices: list[int],
                           movies_df: pd.DataFrame,
                           query_text: str,
                           n_examples: int = 6) -> str:
    """
    Sends representative movies from a cluster to GPT
    and retrieves a short label.

    query_text: the user's original query (for context)
    """
    # select representative movies
    sample_size = min(n_examples, len(cluster_indices))
    sample_idx  = np.random.choice(cluster_indices, size=sample_size, replace=False)

    titles = movies_df.iloc[sample_idx]['title_clean'].tolist()
    genres = movies_df.iloc[sample_idx]['genres_clean'].tolist()

    # build movie list for the prompt
    movie_list = '\n'.join(
        f'- {t} ({g})' for t, g in zip(titles, genres)
    )

    prompt = f"""A user is looking for movies. Their request: "{query_text}"

The following movies form one group of options:
{movie_list}

Write a single short label (4-7 words) that describes what this group of movies has in common.
The label should help the user quickly decide if this group matches what they want.
Do NOT mention specific movie titles. Just the label, nothing else."""

    response = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=20,
        temperature=0.3,
    )

    return response.choices[0].message.content.strip()


print('generate_cluster_label defined')

generate_cluster_label defined


### Part 2 — test on a real cluster

In [ ]:
# build a sample cluster
test_indices = list(range(500))
km = KMeans(n_clusters=5, random_state=42, n_init='auto')
labels = km.fit_predict(embeddings[test_indices])

# split into clusters
partition = {}
for idx, lbl in zip(test_indices, labels):
    if lbl not in partition:
        partition[lbl] = []
    partition[lbl].append(idx)

query = "I want something fun for the whole family"

print(f'Query: "{query}"')
print()
print('Cluster labels from GPT:')
print()

for k, indices in sorted(partition.items()):
    label = generate_cluster_label(indices, movies, query)
    sample = movies.iloc[indices[:3]]['title_clean'].tolist()
    print(f'  Cluster {k} ({len(indices)} items)')
    print(f'  Label  : "{label}"')
    print(f'  Sample : {sample}')
    print()

Query: "I want something fun for the whole family"

Cluster labels from GPT:

  Cluster 0 (129 items)
  Label  : "Family-friendly comedies with heart and humor."
  Sample : ['Toy Story', 'Grumpier Old Men', 'Waiting to Exhale']

  Cluster 1 (97 items)
  Label  : "Not suitable for the whole family"
  Sample : ['Heat', 'Sudden Death', 'GoldenEye']

  Cluster 2 (76 items)
  Label  : "Family-friendly mix of genres."
  Sample : ['Jumanji', 'Tom and Huck', 'Balto']

  Cluster 3 (71 items)
  Label  : "Family-friendly Romantic Comedy Films"
  Sample : ['Sabrina', 'Sense and Sensibility', 'Leaving Las Vegas']

  Cluster 4 (127 items)
  Label  : "Family-friendly drama films."
  Sample : ['Nixon', 'Powder', 'Othello']



### Part 3 — show interaction with LLM labels

In [ ]:
# simulate a full interaction with LLM labels
# shows only what the user sees

from sentence_transformers import SentenceTransformer
import sys
sys.path.append(str(Path('../')))
from query_templates import get_query_for_movie

sbert = SentenceTransformer('all-MiniLM-L6-v2')

def show_interaction_with_labels(target_idx: int, n_turns: int = 2):
    """
    Runs and displays a full interaction with LLM labels.
    For visualization only — uses a simulated user.
    """
    genres     = movies.iloc[target_idx]['genres']
    query_text = get_query_for_movie(genres)
    query_emb  = sbert.encode(query_text, normalize_embeddings=True)

    # retrieval
    scores  = embeddings @ query_emb
    C = np.argsort(scores)[::-1][:500].tolist()
    if target_idx not in C:
        C.append(target_idx)

    target_title = movies.iloc[target_idx]['title_clean']
    print(f'🎯 Target  : {target_title}')
    print(f'🔍 Query   : "{query_text}"')
    print(f'📦 Initial candidates: {len(C)} items')
    print()

    for turn in range(n_turns):
        k = min(4, len(C) // 5 + 2)
        k = max(2, min(k, len(C)))

        km = KMeans(n_clusters=k, random_state=42, n_init='auto')
        lbl_arr = km.fit_predict(embeddings[C])

        partition = {}
        for idx, lbl in zip(C, lbl_arr):
            if lbl not in partition:
                partition[lbl] = []
            partition[lbl].append(idx)

        print(f'── Turn {turn+1} ({len(C)} items) ──────────────────────')
        print('Which of these best matches what you want?')
        print()

        chosen_cluster = None
        for i, (k_lbl, indices) in enumerate(partition.items()):
            label = generate_cluster_label(indices, movies, query_text)
            selected = '✅' if target_idx in indices else '  '
            print(f'  {selected} Option {i+1}: "{label}"  ({len(indices)} movies)')
            if target_idx in indices:
                chosen_cluster = indices

        print()
        print(f'  → User selects option containing target')
        print()
        C = chosen_cluster

    print(f'── Final candidate set: {len(C)} items ──────────────────')
    for idx in C[:5]:
        marker = '🎯' if idx == target_idx else '  '
        print(f'  {marker} {movies.iloc[idx]["title_clean"]}')
    if len(C) > 5:
        print(f'  ... and {len(C)-5} more')


# test on a single movie
np.random.seed(42)
show_interaction_with_labels(target_idx=0, n_turns=2)  # Toy Story

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

🎯 Target  : Toy Story
🔍 Query   : "I need something cheerful and light"
📦 Initial candidates: 501 items

── Turn 1 (501 items) ──────────────────────
Which of these best matches what you want?

     Option 1: "Musical and cheerful family-friendly films."  (40 movies)
  ✅ Option 2: "Light-hearted comedy film options."  (160 movies)
     Option 3: "Romantic Drama Films"  (189 movies)
     Option 4: "Feel-good romantic comedy films."  (112 movies)

  → User selects option containing target

── Turn 2 (160 items) ──────────────────────
Which of these best matches what you want?

     Option 1: "Feel-good comedy films with light-hearted humor."  (69 movies)
     Option 2: "Cheerful and light comedies and comedy dramas."  (51 movies)
     Option 3: "Feel-good comedy films with light-hearted humor."  (18 movies)
  ✅ Option 4: "Cheerful and light children's comedy films"  (22 movies)

  → User selects option containing target

── Final candidate set: 22 items ──────────────────
     Flubber
  

### Part 4 — save example interactions for thesis

In [ ]:
# example interactions for the qualitative analysis section of the thesis
examples = [
    0,    # Toy Story
    50,   # another movie
    200,  # another movie
]

for target in examples:
    print('=' * 60)
    show_interaction_with_labels(target_idx=target, n_turns=2)
    print()

print('✅ notebook 05 complete')

🎯 Target  : Toy Story
🔍 Query   : "I need something cheerful and light"
📦 Initial candidates: 501 items

── Turn 1 (501 items) ──────────────────────
Which of these best matches what you want?

     Option 1: "Cheerful and whimsical children's adventures."  (40 movies)
  ✅ Option 2: "Feel-good comedy films"  (160 movies)
     Option 3: "Drama Romance films"  (189 movies)
     Option 4: "Cheerful and light romantic comedies."  (112 movies)

  → User selects option containing target

── Turn 2 (160 items) ──────────────────────
Which of these best matches what you want?

     Option 1: "Upbeat and humorous comedy films"  (69 movies)
     Option 2: "Feel-good comedy films with light-hearted humor"  (51 movies)
     Option 3: "Cheerful and light comedy films."  (18 movies)
  ✅ Option 4: "Cheerful and light children's comedies."  (22 movies)

  → User selects option containing target

── Final candidate set: 22 items ──────────────────
     Flubber
     Candleshoe
     Home Alone
     Jingl